In [11]:
import os
if os.name == 'nt': # Check if windows
    os.add_dll_directory(r'C:\Program Files\SuperTuxKart 1.5')

import multiprocessing as mp
import numpy as np

import torch
import torch.optim as optim
import torch.nn.functional as F
from actor import ActorNetwork
from critic import CriticNetwork

In [12]:
'''
Explanation for state_dim=52:
TrackErrorDot = 1 (Distance from center of track)
TrackErrorCross = 1 (Left or Right)
Velocity = 3
HeadingAlignmentDot = 1 (Dot product of front vector and track vector)
HeadingAlignmentCross = 1 (Left or Right)
Jumping = 1
Rotation = 4
Distance = 1
Total = 13
Frame Stacking (x4) = 52
'''

import os
if os.name == 'nt': #Check if windows
    os.add_dll_directory(r'C:\Program Files\SuperTuxKart 1.5')
import pystk2
import numpy as np
from collections import deque

class ProcessState:
	def __init__(self, PathNodes, max_speed=30, map_size=100, track_length=2000):
		self.frame = deque(maxlen=4)    # Window holding last 4 frames
		self.max_speed = max_speed
		self.map_size = map_size
		self.track_length = track_length
		self.PathNodes = np.array(PathNodes)[:,0,:]
		self.LookaheadIndex = 5

	def processObservation(self, obs):
		KartLocation = np.array(obs['location'], dtype=np.float32)
		KartFrontLocation = np.array(obs['front'], dtype=np.float32)
		SquaredNodes = np.sum((self.PathNodes-KartLocation)**2, axis=1)
		AnchorNodeIndex = np.argmin(SquaredNodes)
		AnchorNode = self.PathNodes[AnchorNodeIndex]
		TargetNode = self.PathNodes[(AnchorNodeIndex+self.LookaheadIndex)%len(self.PathNodes)]

		OffsetVector = KartLocation - AnchorNode
		TrackErrorDot = np.array([SquaredNodes[AnchorNodeIndex]], dtype=np.float32)

		TargetVector = (TargetNode-AnchorNode)/(np.linalg.norm(TargetNode-AnchorNode)+1e-6) #Divide with magnitude to obtain direction unit vector
		TrackErrorCross = np.array([TargetVector[0]*OffsetVector[2] - TargetVector[2]*OffsetVector[0]], dtype=np.float32)
		FrontVector = (KartFrontLocation-KartLocation)/(np.linalg.norm(KartFrontLocation-KartLocation)+1e-6)
		HeadingAlignmentDot = np.array([np.dot(TargetVector,FrontVector)], dtype=np.float32)
		HeadingAlignmentCross = np.array([TargetVector[0]*FrontVector[2] - TargetVector[2]*FrontVector[0]], dtype=np.float32)

		vel = np.array(obs["velocity"], dtype=np.float32) / self.max_speed
		vel = np.clip(vel, -1.0, 1.0)

		jump = np.array([1.0 if obs["jumping"] else 0.0], dtype=np.float32)

		rotation = np.array(obs["rotation"], dtype=np.float32)

		dist = np.array([obs.get("distance_down_track", 0.0)], dtype=np.float32) / self.track_length

		state = np.concatenate([TrackErrorDot, TrackErrorCross, vel, HeadingAlignmentDot, HeadingAlignmentCross, jump, rotation, dist])

		if len(self.frame) == 0:
			for _ in range(4):
				self.frame.append(state)
		else:
			self.frame.append(state)

		return (np.concatenate(self.frame),TrackErrorDot[0],HeadingAlignmentDot[0])

def SingleInstance(rank,pipe):
	pystk2.init(pystk2.GraphicsConfig.hd())
	WorldState = pystk2.WorldState()
	config = pystk2.RaceConfig(track='lighthouse', num_kart=1, laps=1)
	config.players[0].controller = pystk2.PlayerConfig.Controller.PLAYER_CONTROL
	race = pystk2.Race(config)
	try:
		race.start()
			
		# Track details must be loaded after race start
		track = pystk2.Track()
		track.update()
		track_length = track.length
		max_coordinate = np.max(np.abs(track.path_nodes))
		
		processor = ProcessState(max_speed=30, map_size=max_coordinate, track_length=track_length, PathNodes=track.path_nodes)
		RaceEnded = False
		reward = 0.0

		WorldState.update()
			
		kart = WorldState.karts[0]
		prev_dist = kart.distance_down_track
		
		obs = {
			"location": kart.location,
			"velocity": kart.velocity,
			"front": kart.front,
			"jumping": kart.jumping,
			"rotation": kart.rotation,
			"distance_down_track": prev_dist
		}
		np_obs,TrackError,HeadingAlignment = processor.processObservation(obs=obs)
		#Send observation to model
		pipe.send([np_obs,reward,RaceEnded])

		StuckFrameCounter = 0

		while True:
			#Receive action from model
			ActionMessage = pipe.recv()

			if ActionMessage == 'TERMINATE':
				return

			action = pystk2.Action()
			action.steer = ActionMessage[0]
			action.acceleration = ActionMessage[1]
			action.brake = True if ActionMessage[2] > 0.5 else False

			# Step the environment
			RaceEnded = not race.step(action)

			WorldState.update()

			kart = WorldState.karts[0]
			current_dist = kart.distance_down_track
			
			obs = {
				"location": kart.location,
				"velocity": kart.velocity,
				"front": kart.front,
				"jumping": kart.jumping,
				"rotation": kart.rotation,
				"distance_down_track": current_dist
			}
			np_obs,TrackError,HeadingAlignment = processor.processObservation(obs=obs)

			# Reward Calculation
			vel_x, vel_y, vel_z = obs['velocity']
			speed = (vel_x**2 + vel_y**2 + vel_z**2)**0.5
			
			delta_dist = current_dist - prev_dist
			if abs(delta_dist) > 20.0:
				delta_dist = 0.0		# New lap

			reward = delta_dist * 1000.0
			reward -= min((TrackError * 5),100)
			reward += (HeadingAlignment * 100.0)
				
			if current_dist < 0:
				reward -= 10.0
				
			prev_dist = current_dist

			if speed < 2.0:
				StuckFrameCounter += 1
			else:
				StuckFrameCounter = 0

			if StuckFrameCounter > 60:
				reward = -200.0
				RaceEnded = True
							
			#Send observation to model
			pipe.send([np_obs,float(reward),RaceEnded])

	finally:
		# Critical Cleanup
		race.stop()
		del race
		pystk2.clean()


In [13]:
# --- STEP 1: THE PPO UPDATE FUNCTION ---

# Instantiate models with an input dimension of 52
actor_net = ActorNetwork(state_dim=52)
critic_net = CriticNetwork(state_dim=52)

# Set to evaluation mode for simulation
actor_net.eval()
critic_net.eval()

# Initialize Optimizer for both networks
optimizer = optim.Adam([
	{'params': actor_net.parameters(), 'lr': 1e-4},
	{'params': critic_net.parameters(), 'lr': 1e-4}
])

In [14]:
def main():
	try:
		EpochLimit = 1

		for episode in range(EpochLimit):
			buffer = []
			total_episode_reward = 0.0
			ProcessList = []
			ConList = []
			for i in range(1):
				ParentCon,ChildCon = mp.Pipe()
				process = mp.Process(target=SingleInstance,args=(i,ChildCon))
				ProcessList.append(process)
				ConList.append(ParentCon)
				process.start()
			BatchStates = []
			BatchDones = []

			for con in ConList:
					np_obs,reward,RaceDone = con.recv()
					BatchStates.append(np_obs)
					BatchDones.append(RaceDone)

			for step in range(1000):

				# --- Phase 2 Brain Injection ---
				state_tensor = torch.FloatTensor(np.array(BatchStates))
				
				# Sanity Check for incoming engine observations
				if torch.isnan(state_tensor).any():
					print("NaN detected in engine observations! Terminating episode.")
					break
					
				with torch.no_grad():
					action_dist = actor_net(state_tensor)
					sampled_action = action_dist.sample()
					state_value = critic_net(state_tensor)

					BatchLogProbs = action_dist.log_prob(sampled_action).sum(dim=-1)
				
				MemoryActions = []

				for i,con in enumerate(ConList):
					steer_val = torch.clamp(sampled_action[i, 0], min=-1.0, max=1.0).item()
					accel_val = torch.clamp(sampled_action[i, 1], min=0.0, max=1.0).item()
					BrakeVal = torch.clamp(sampled_action[i,2],min=0.0,max=1.0).item()
					MemoryActions.append((steer_val,accel_val,BrakeVal))

					if BatchDones[i]:
						con.send((0.0,0.0,0.0)) #Dummy vals
					else:
						con.send((steer_val,accel_val,BrakeVal))

				NextStates = []
				PreviousRewards = []
				PreviousDones = []

				for con in ConList:
						np_obs,reward,RaceDone = con.recv()
						NextStates.append(np_obs)
						PreviousRewards.append(reward)
						PreviousDones.append(RaceDone)
				
				total_episode_reward += sum(PreviousRewards)

				BatchStates = NextStates
				BatchDones = PreviousDones

				if all(PreviousDones):
					break
						
			# --- END OF EPISODE TRIGGER ---
			print(f"Episode: {episode + 1}/{EpochLimit} | Total Reward: {(total_episode_reward/max(len(buffer),1)):.2f} | Buffer Size: {len(buffer)}")

			for con in ConList:
				con.send('TERMINATE')

			for process in ProcessList:
				process.join()
	
	finally:
		for con in ConList:
			con.send('TERMINATE')

		for process in ProcessList:
			process.join()

In [15]:
if __name__ == '__main__':
	main()

..:: Antarctica Rendering Engine 2.0 ::..
Episode: 1/1 | Total Reward: 104053.66 | Buffer Size: 0


  wl_callback#51 still attached
  wl_surface#36 still attached
